# Experiment A — Original 58-RNA three-predictor workflow

**Project 7 | RNA Bioinformatics | Université Paris-Saclay**

This notebook preserves the **actual Python workflows from the original executed Colab notebook** (`Untitled11.ipynb`, provided during the project). It includes computation of nucleotide pairing probabilities using **ViennaRNA, LinearPartition-V, and EternaFold**, comparison to PDB-derived reference labels, reliability plots, and a grouped bootstrap.

**Provenance and status:** Original code was recovered and ordered into a clean workflow. Earlier saved outputs document successful execution of the original cells, but **this curated copy has not been run end-to-end or independently validated to reproduce byte-identical prediction outputs**. One abandoned LinearPartition executable attempt and redundant setup cell were omitted. The predictor Git commits have been pinned to those recorded in the original executed notebook.

**Inputs:** At the upload prompt use `Project7_ExperimentA_58_screening_package.zip`, containing the 58-sequence FASTA, per-nucleotide experimental pairing labels, the supplement-section manifest and source CT annotations. **Do not publish third-party reference inputs without checking redistribution permissions.**

**Output:** Per-nucleotide pairing probabilities for 58 RNAs (3426 nucleotides) and the 3-predictor matched/summary tables. Accuracy and independence limitations are documented in `Experiment_A/README.md` and `Experiment_A/scripts/REPRODUCIBILITY_STATUS.md`.

**Notebook usage:** Open in Google Colab, choose `Runtime → Run all`, and upload the input ZIP when prompted. Compilation and prediction may be compute-intensive. This notebook does not implement the later 50/45-RNA sensitivity and post-hoc calibration analyses; those have their own scripts in the repository.

## 1. Install analysis dependencies

In [ ]:
%pip -q install ViennaRNA==2.8.0 pandas numpy matplotlib
import RNA
print("ViennaRNA version:", RNA.__version__)


## 2. Upload the structural-reference input package and load 58 RNAs

In [ ]:

from google.colab import files
import zipfile
from pathlib import Path

# Choose your Experiment A ZIP file
uploaded = files.upload()

zip_name = next(
    name for name in uploaded
    if name.endswith(".zip")
)

# Extract all experiment files
with zipfile.ZipFile(zip_name) as z:
    z.extractall("experiment_a")

# Count RNA sequences
fasta = Path("experiment_a/Project7_ExperimentA_58_sequences.fasta")
sequences = fasta.read_text()

rna_count = sequences.count(">")

print("SUCCESS: Experiment A files uploaded!")
print("Number of RNA sequences:", rna_count)


In [ ]:

import RNA
import csv
from pathlib import Path

# Find the uploaded FASTA file
fasta = next(Path("experiment_a").rglob("*58_sequences.fasta"))

# Read the 58 RNA sequences
sequences = {}
name = None

for line in fasta.read_text().splitlines():
    line = line.strip()
    if not line:
        continue
    if line.startswith(">"):
        name = line[1:].split()[0]
        sequences[name] = ""
    else:
        sequences[name] += line.upper()

print("RNA sequences found:", len(sequences))
assert len(sequences) == 58

# Calculate pairing probabilities
results = []

for name, sequence in sequences.items():
    fc = RNA.fold_compound(sequence)
    fc.pf()
    probabilities = fc.bpp(symmetric=True)

    for i, nucleotide in enumerate(sequence, start=1):
        p_paired = sum(
            probabilities[i][j]
            for j in range(1, len(sequence) + 1)
            if j != i
        )

        assert -1e-6 <= p_paired <= 1 + 1e-6

        results.append([
            name, i, nucleotide,
            max(0.0, min(1.0, p_paired))
        ])

    print("Completed:", name)

# Save predictions
filename = "ExperimentA_ViennaRNA_58_predictions.csv"

with open(filename, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "structure_id",
        "position",
        "nucleotide",
        "predicted_paired_probability"
    ])
    writer.writerows(results)

print("\nSUCCESS!")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("ViennaRNA version:", RNA.__version__)
print("Saved file:", filename)


## 3. ViennaRNA base-pair probabilities and calibration

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the ViennaRNA predictions you just generated
pred = pd.read_csv("ExperimentA_ViennaRNA_58_predictions.csv")

# Load experimental reference labels from your ZIP
reference_file = next(
    Path("experiment_a").rglob(
        "Project7_ExperimentA_58_reference_labels.csv"
    )
)
ref = pd.read_csv(reference_file)

# Match RNA IDs and nucleotide positions
keys = ["structure_id", "position"]

assert not pred.duplicated(keys).any()
assert not ref.duplicated(keys).any()

data = pred.merge(
    ref[keys + ["nucleotide", "reference_paired_label"]],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_pred", "_ref")
)

assert len(data) == 3426
assert (data["_merge"] == "both").all()
assert (data["nucleotide_pred"] == data["nucleotide_ref"]).all()

# Predicted probabilities and experimental labels
p = data["predicted_paired_probability"].to_numpy(float)
y = data["reference_paired_label"].to_numpy(int)

assert np.isfinite(p).all()
assert ((p >= 0) & (p <= 1)).all()
assert set(y).issubset({0, 1})

# Brier score
brier = np.mean((p - y) ** 2)

# Log loss
p_safe = np.clip(p, 1e-12, 1 - 1e-12)
logloss = -np.mean(
    y * np.log(p_safe) +
    (1 - y) * np.log(1 - p_safe)
)

# ECE: 10 equal-width probability bins
bin_ids = np.minimum((p * 10).astype(int), 9)
ece = 0.0
bin_rows = []

for b in range(10):
    mask = bin_ids == b
    if not mask.any():
        continue

    confidence = p[mask].mean()
    observed = y[mask].mean()
    count = mask.sum()

    ece += (count / len(y)) * abs(confidence - observed)

    bin_rows.append({
        "bin": b + 1,
        "count": int(count),
        "mean_predicted": confidence,
        "observed_fraction_paired": observed
    })

# Save reproducible results
data.to_csv("ExperimentA_ViennaRNA_58_matched.csv", index=False)
pd.DataFrame(bin_rows).to_csv(
    "ExperimentA_ViennaRNA_58_calibration_bins.csv",
    index=False
)

print("SUCCESS: CALIBRATION CALCULATED")
print("RNAs:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print("ECE (10 bins):", round(ece, 4))
print("Brier score:", round(brier, 4))
print("Log loss:", round(logloss, 4))


## 4. LinearPartition-V, fixed Git revision and beam size 100

In [ ]:
import subprocess
from pathlib import Path

# Download the official LinearPartition source code
if not Path("LinearPartition/.git").exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/LinearFold/LinearPartition.git",
        "LinearPartition"
    ], check=True)

# Match the exact recorded analysis revision; do not silently use a newer HEAD.
subprocess.run([
    "git", "-C", "LinearPartition", "checkout",
    "b450fb3e63189073b68d385589035f992080aa3a"
], check=True)

# Compile LinearPartition
subprocess.run([
    "make", "-C", "LinearPartition",
    "linearpartition", "-j2"
], check=True)

# Check that LinearPartition-V is ready
executable = Path("LinearPartition/bin/linearpartition_v")

assert executable.is_file(), "LinearPartition-V was not built"

commit = subprocess.check_output([
    "git", "-C", "LinearPartition", "rev-parse", "HEAD"
], text=True).strip()

print("SUCCESS: LinearPartition-V is ready!")
print("Source revision:", commit)

In [ ]:

import subprocess
import sys
from pathlib import Path

# Correct LinearPartition launcher
launcher = Path("LinearPartition/linearpartition").resolve()

# Test only our first RNA
sequence = sequences["1f1t_A"]

output_folder = Path("LinearPartition_58_outputs")
output_folder.mkdir(exist_ok=True)

output_file = (output_folder / "1f1t_A.bpp").resolve()

run = subprocess.run(
    [
        sys.executable,
        str(launcher),
        "-V",
        "-b", "100",
        "-r", str(output_file)
    ],
    input=sequence + "\n",
    text=True,
    capture_output=True
)

print("Return code:", run.returncode)
print("Program output:", run.stdout)
print("Program messages:", run.stderr)
print("Probability file created:", output_file.exists())

if run.returncode == 0 and output_file.exists():
    lines = output_file.read_text().splitlines()
    print("SUCCESS: LinearPartition-V works!")
    print("Number of base-pair probability entries:", len(lines))
    print("First few entries:")
    print("\n".join(lines[:5]))


In [ ]:

import subprocess
import sys
import csv
from pathlib import Path

# Use the working LinearPartition launcher
launcher = Path("LinearPartition/linearpartition").resolve()
output_folder = Path("LinearPartition_58_outputs")
output_folder.mkdir(exist_ok=True)

assert len(sequences) == 58

results = []
raw_results = []
corrections = 0

for name, sequence in sequences.items():

    bpp_file = (output_folder / f"{name}.bpp").resolve()

    # Remove any previous output to avoid stale results
    if bpp_file.exists():
        bpp_file.unlink()

    run = subprocess.run(
        [
            sys.executable,
            str(launcher),
            "-V",
            "-b", "100",
            "-r", str(bpp_file)
        ],
        input=sequence + "\n",
        text=True,
        capture_output=True
    )

    if run.returncode != 0 or not bpp_file.exists():
        raise RuntimeError(
            f"Failed for {name}\n{run.stdout}\n{run.stderr}"
        )

    paired = [0.0] * (len(sequence) + 1)

    for line in bpp_file.read_text().splitlines():
        parts = line.split()

        if not parts:
            continue

        if len(parts) != 3:
            raise ValueError(f"Unexpected line: {line}")

        i, j = int(parts[0]), int(parts[1])
        probability = float(parts[2])

        assert 1 <= i < j <= len(sequence)
        assert 0 <= probability <= 1

        paired[i] += probability
        paired[j] += probability

    for position, nucleotide in enumerate(sequence, 1):

        raw_p = paired[position]

        # Preserve raw values; allow only tiny corrections
        if not -0.001 <= raw_p <= 1.001:
            raise ValueError(
                f"Probability out of range: {name}, "
                f"position {position}: {raw_p}"
            )

        corrected_p = min(1.0, max(0.0, raw_p))

        if corrected_p != raw_p:
            corrections += 1

        raw_results.append([
            name, position, nucleotide, raw_p
        ])

        results.append([
            name, position, nucleotide, corrected_p
        ])

    print("Completed:", name)

columns = [
    "structure_id",
    "position",
    "nucleotide",
    "predicted_paired_probability"
]

for filename, rows in [
    ("ExperimentA_LinearPartitionV_58_predictions.csv", results),
    ("ExperimentA_LinearPartitionV_58_predictions_raw.csv", raw_results)
]:
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)

print("\nSUCCESS: LinearPartition-V completed!")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("Small numerical corrections:", corrections)
print("Prediction CSV files saved!")


In [ ]:

import pandas as pd
import numpy as np

# Load predictions from the two predictors
vienna = pd.read_csv("ExperimentA_ViennaRNA_58_matched.csv")
linear = pd.read_csv("ExperimentA_LinearPartitionV_58_predictions.csv")

keys = ["structure_id", "position"]

# Match both methods to the same experimental labels
data = vienna[
    keys + [
        "nucleotide_ref",
        "reference_paired_label",
        "predicted_paired_probability"
    ]
].merge(
    linear[keys + ["nucleotide", "predicted_paired_probability"]],
    on=keys,
    validate="one_to_one",
    how="outer",
    indicator=True,
    suffixes=("_vienna", "_linear")
)

# Quality checks: no missing positions or sequence mismatches
assert len(data) == 3426
assert (data["_merge"] == "both").all()
assert (data["nucleotide_ref"] == data["nucleotide"]).all()

y = data["reference_paired_label"].to_numpy(int)

def calculate_metrics(p, y):
    p = np.asarray(p, dtype=float)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe) +
        (1 - y) * np.log(1 - safe)
    )

    bin_ids = np.minimum((p * 10).astype(int), 9)
    ece = 0.0
    rows = []

    for b in range(10):
        mask = bin_ids == b
        if mask.any():
            mean_p = p[mask].mean()
            observed = y[mask].mean()
            ece += mask.mean() * abs(mean_p - observed)

            rows.append({
                "bin": b + 1,
                "count": int(mask.sum()),
                "mean_predicted": mean_p,
                "observed_fraction_paired": observed
            })

    return (ece, brier, logloss), pd.DataFrame(rows)

v_metrics, _ = calculate_metrics(
    data["predicted_paired_probability_vienna"], y
)

l_metrics, l_bins = calculate_metrics(
    data["predicted_paired_probability_linear"], y
)

print("SUCCESS: TWO PREDICTORS COMPARED")
print("RNAs:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print()

print("                  ViennaRNA   LinearPartition-V")
for i, metric in enumerate(["ECE", "Brier score", "Log loss"]):
    print(f"{metric:12s}      {v_metrics[i]:.4f}       {l_metrics[i]:.4f}")

# Save results for later graphs and analysis
data.to_csv("ExperimentA_Vienna_vs_Linear_matched.csv", index=False)

l_bins.to_csv(
    "ExperimentA_LinearPartitionV_58_calibration_bins.csv",
    index=False
)

print("\nResults saved.")


## 5. EternaFold posterior probabilities, fixed Git revision

In [ ]:
import subprocess
from pathlib import Path

# Download EternaFold source code
if not Path("EternaFold/.git").exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/WaymentSteeleLab/EternaFold.git"],
        check=True
    )

# Match the exact recorded analysis revision; do not silently use a newer HEAD.
subprocess.run([
    "git", "-C", "EternaFold", "checkout",
    "87b9aac55cee14fd562049d08f7b92d3131f10ce"
], check=True)

# Compile the single-process EternaFold program
subprocess.run(
    ["make", "-C", "EternaFold/src", "-j2"],
    check=True
)

# Verify installation
exe = Path("EternaFold/src/contrafold")
params = Path("EternaFold/parameters/EternaFoldParams.v1")

assert exe.is_file(), "EternaFold executable missing"
assert params.is_file(), "EternaFold parameters missing"

revision = subprocess.check_output(
    ["git", "-C", "EternaFold", "rev-parse", "HEAD"],
    text=True
).strip()

print("\nSUCCESS: EternaFold installed!")
print("Source revision:", revision)

In [ ]:

import subprocess
from pathlib import Path

# Use the first RNA from your existing dataset
sequence = sequences["1f1t_A"]

# Create EternaFold input file
input_file = Path("eterna_test.seq").resolve()
input_file.write_text(sequence + "\n")

# Define the program and its parameters
exe = Path("EternaFold/src/contrafold").resolve()
params = Path("EternaFold/parameters/EternaFoldParams.v1").resolve()
output_file = Path("eterna_test_probabilities.txt").resolve()

# Calculate base-pair probabilities
run = subprocess.run(
    [
        str(exe), "predict", str(input_file),
        "--params", str(params),
        "--posteriors", "0.00001", str(output_file)
    ],
    capture_output=True,
    text=True
)

print("Return code:", run.returncode)
print("Probability file created:", output_file.exists())

if run.returncode != 0:
    print("Error:", run.stderr)
elif output_file.exists():
    lines = output_file.read_text().splitlines()
    print("SUCCESS: EternaFold works!")
    print("RNA length:", len(sequence))
    print("Output rows:", len(lines))
    print("First three rows:")
    print("\n".join(lines[:3]))


In [ ]:

import subprocess
import csv
from pathlib import Path

# EternaFold program and parameters
exe = Path("EternaFold/src/contrafold").resolve()
params = Path("EternaFold/parameters/EternaFoldParams.v1").resolve()

folder = Path("EternaFold_58_outputs")
folder.mkdir(exist_ok=True)

assert len(sequences) == 58

results = []
raw_results = []
corrections = 0

for name, sequence in sequences.items():

    input_file = (folder / f"{name}.seq").resolve()
    output_file = (folder / f"{name}.bpp").resolve()

    input_file.write_text(sequence + "\n")

    # Prevent accidentally using an old output file
    if output_file.exists():
        output_file.unlink()

    run = subprocess.run(
        [
            str(exe), "predict", str(input_file),
            "--params", str(params),
            "--posteriors", "0.00001",
            str(output_file)
        ],
        capture_output=True,
        text=True
    )

    if run.returncode != 0 or not output_file.exists():
        raise RuntimeError(
            f"Failed for {name}\n{run.stdout}\n{run.stderr}"
        )

    paired = [0.0] * (len(sequence) + 1)
    lines = output_file.read_text().splitlines()

    assert len(lines) == len(sequence), name

    for expected_i, line in enumerate(lines, start=1):
        parts = line.split()

        i = int(parts[0])
        nucleotide = parts[1]

        assert i == expected_i
        assert nucleotide == sequence[i - 1]

        for entry in parts[2:]:
            j_text, p_text = entry.split(":")
            j = int(j_text)
            probability = float(p_text)

            assert 1 <= i < j <= len(sequence)
            assert 0 <= probability <= 1

            # Each pair is listed once, so add to both ends
            paired[i] += probability
            paired[j] += probability

    for position, nucleotide in enumerate(sequence, 1):
        raw_p = paired[position]

        # Only tiny numerical deviations may be corrected
        if not -0.001 <= raw_p <= 1.001:
            raise ValueError(
                f"Probability out of range: {name}, "
                f"position {position}: {raw_p}"
            )

        corrected_p = min(1.0, max(0.0, raw_p))

        if corrected_p != raw_p:
            corrections += 1

        raw_results.append([
            name, position, nucleotide, raw_p
        ])

        results.append([
            name, position, nucleotide, corrected_p
        ])

    print("Completed:", name)

columns = [
    "structure_id",
    "position",
    "nucleotide",
    "predicted_paired_probability"
]

for filename, rows in [
    ("ExperimentA_EternaFold_58_predictions.csv", results),
    ("ExperimentA_EternaFold_58_predictions_raw.csv", raw_results)
]:
    with open(filename, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(columns)
        writer.writerows(rows)

print("\nSUCCESS: EternaFold completed!")
print("RNA structures:", len(sequences))
print("Nucleotide predictions:", len(results))
print("Small numerical corrections:", corrections)
print("Prediction CSV files saved!")


## 6. Compare all three predictors and generate reliability plot

In [ ]:

import pandas as pd
import numpy as np

# Load the existing two-predictor comparison
base = pd.read_csv(
    "ExperimentA_Vienna_vs_Linear_matched.csv"
)

# Load EternaFold's new predictions
eterna = pd.read_csv(
    "ExperimentA_EternaFold_58_predictions.csv"
)

keys = ["structure_id", "position"]

assert not base.duplicated(keys).any()
assert not eterna.duplicated(keys).any()

# Rename columns to make the three methods clear
base = base.rename(columns={
    "predicted_paired_probability_vienna": "p_vienna",
    "predicted_paired_probability_linear": "p_linear"
})

eterna = eterna.rename(columns={
    "nucleotide": "nucleotide_eterna",
    "predicted_paired_probability": "p_eterna"
})

# Match all methods by RNA ID and nucleotide position
data = base.merge(
    eterna[keys + ["nucleotide_eterna", "p_eterna"]],
    on=keys,
    how="outer",
    validate="one_to_one",
    indicator="eterna_match"
)

# Verify that all three methods used identical RNA positions
assert len(data) == 3426
assert (data["eterna_match"] == "both").all()
assert (data["nucleotide_ref"] == data["nucleotide_eterna"]).all()
assert (data["nucleotide_ref"] == data["nucleotide"]).all()

y = data["reference_paired_label"].to_numpy(int)

# Calculate ECE, Brier score and log loss
def metrics(p, y):
    p = np.asarray(p, dtype=float)

    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()

    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe)
        + (1 - y) * np.log(1 - safe)
    )

    bin_id = np.minimum((p * 10).astype(int), 9)
    ece = 0.0

    for b in range(10):
        mask = bin_id == b
        if mask.any():
            ece += mask.mean() * abs(
                p[mask].mean() - y[mask].mean()
            )

    return ece, brier, logloss

results = []

for method, column in [
    ("ViennaRNA", "p_vienna"),
    ("LinearPartition-V", "p_linear"),
    ("EternaFold", "p_eterna")
]:
    ece, brier, logloss = metrics(data[column], y)

    results.append({
        "Predictor": method,
        "ECE": ece,
        "Brier": brier,
        "LogLoss": logloss
    })

summary = pd.DataFrame(results)

print("SUCCESS: ALL THREE PREDICTORS COMPARED")
print("RNA structures:", data["structure_id"].nunique())
print("Nucleotides:", len(data))
print()
print(summary.round(4).to_string(index=False))

# Save everything
summary.to_csv(
    "ExperimentA_three_predictor_summary.csv",
    index=False
)
data.to_csv(
    "ExperimentA_three_predictor_matched.csv",
    index=False
)

print("\nResults saved successfully!")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

data = pd.read_csv(
    "ExperimentA_three_predictor_matched.csv"
)

y = data["reference_paired_label"].to_numpy(int)

methods = {
    "ViennaRNA": "p_vienna",
    "LinearPartition-V": "p_linear",
    "EternaFold": "p_eterna"
}

plt.figure(figsize=(8, 7))
plt.plot([0, 1], [0, 1], "k--",
         label="Perfect calibration")

for method, column in methods.items():
    p = data[column].to_numpy(float)
    bins = np.minimum((p * 10).astype(int), 9)

    x_values = []
    observed_values = []

    for b in range(10):
        mask = bins == b
        if mask.sum() > 0:
            x_values.append(p[mask].mean())
            observed_values.append(y[mask].mean())

    plt.plot(
        x_values, observed_values,
        "o-", linewidth=2, label=method
    )

plt.xlabel("Mean predicted pairing probability")
plt.ylabel("Observed fraction paired")
plt.title("Experiment A: Three-Predictor Reliability")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ExperimentA_three_predictor_reliability.png",
    dpi=300
)
plt.show()

print("SUCCESS: Three-predictor graph saved!")

## 7. Paired supplement-section bootstrap (35 operational groups, not certified evolutionary families)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files

# Load results from all three predictors
data = pd.read_csv(
    "ExperimentA_three_predictor_matched.csv"
)

# Load published RNA grouping information
manifest_file = next(
    Path("experiment_a").rglob(
        "Project7_ExperimentA_58_manifest.csv"
    )
)

manifest = pd.read_csv(
    manifest_file,
    dtype={"supp_section": str}
)

data = data.merge(
    manifest[["structure_id", "supp_section"]],
    on="structure_id",
    validate="many_to_one"
).reset_index(drop=True)

assert len(data) == 3426
assert data["supp_section"].notna().all()

labels = data["reference_paired_label"].to_numpy(int)

methods = {
    "ViennaRNA": data["p_vienna"].to_numpy(float),
    "LinearPartition-V": data["p_linear"].to_numpy(float),
    "EternaFold": data["p_eterna"].to_numpy(float)
}

def scores(p, y):
    brier = np.mean((p - y) ** 2)

    safe = np.clip(p, 1e-12, 1 - 1e-12)
    logloss = -np.mean(
        y * np.log(safe) +
        (1 - y) * np.log(1 - safe)
    )

    bin_id = np.minimum((p * 10).astype(int), 9)
    ece = 0.0

    for b in range(10):
        mask = bin_id == b
        if mask.any():
            ece += mask.mean() * abs(
                p[mask].mean() - y[mask].mean()
            )

    return np.array([ece, brier, logloss])

# Group related RNA structures together
groups = list(
    data.groupby("supp_section").indices.values()
)

print("RNAs:", data["structure_id"].nunique())
print("Published groups:", len(groups))

# Paired bootstrap: same sampled groups for all methods
rng = np.random.default_rng(42)
boot = {name: [] for name in methods}

for repeat in range(2000):
    selected = rng.integers(
        0, len(groups), size=len(groups)
    )

    idx = np.concatenate([groups[i] for i in selected])

    for name, p in methods.items():
        boot[name].append(scores(p[idx], labels[idx]))

for name in methods:
    boot[name] = np.array(boot[name])

metric_names = ["ECE", "Brier", "Log loss"]
rows = []

print("\n95% GROUPED BOOTSTRAP INTERVALS")

for name, p in methods.items():
    observed = scores(p, labels)
    lower, upper = np.percentile(
        boot[name], [2.5, 97.5], axis=0
    )

    print("\n" + name)

    for j, metric in enumerate(metric_names):
        print(
            f"{metric}: {observed[j]:.4f} "
            f"[{lower[j]:.4f}, {upper[j]:.4f}]"
        )

        rows.append({
            "predictor": name,
            "metric": metric,
            "estimate": observed[j],
            "CI_lower": lower[j],
            "CI_upper": upper[j]
        })

# Compare EternaFold directly against both alternatives
difference_rows = []

print("\nPAIRED ADVANTAGE OF ETERNAFOLD")
print("Positive numbers favor EternaFold.")

for other in ["ViennaRNA", "LinearPartition-V"]:
    difference = (
        scores(methods[other], labels)
        - scores(methods["EternaFold"], labels)
    )

    boot_difference = boot[other] - boot["EternaFold"]
    lower, upper = np.percentile(
        boot_difference, [2.5, 97.5], axis=0
    )

    print("\nEternaFold versus", other)

    for j, metric in enumerate(metric_names):
        print(
            f"{metric} improvement: {difference[j]:.4f} "
            f"[{lower[j]:.4f}, {upper[j]:.4f}]"
        )

        difference_rows.append({
            "comparison": f"{other} minus EternaFold",
            "metric": metric,
            "improvement": difference[j],
            "CI_lower": lower[j],
            "CI_upper": upper[j]
        })

pd.DataFrame(rows).to_csv(
    "ExperimentA_three_predictor_bootstrap.csv",
    index=False
)

pd.DataFrame(difference_rows).to_csv(
    "ExperimentA_paired_differences.csv",
    index=False
)

print("\nSUCCESS: Statistical comparison completed!")

files.download("ExperimentA_three_predictor_bootstrap.csv")
files.download("ExperimentA_paired_differences.csv")

## Notes on interpretation and reproducing other analyses

- **No family-disjoint guarantee:** The 35 supplement-section labels are grouping proxies, not certified evolutionary families. Rfam negative hits also do not establish training independence.
- **Pseudoknots:** 31 of the original 89 structures had crossing annotations and were excluded from the 58-RNA reference comparison.
- **Prediction settings:** ViennaRNA Python partition function; LinearPartition thermodynamic `-V`, beam `100`, base-pair probability output; EternaFold `EternaFoldParams.v1`, `contrafold predict --posteriors 0.00001`.
- **Later analyses:** 50/45-RNA sensitivity tests, 5,000 grouped bootstraps, and section-held-out temperature/isotonic fits are implemented **separately** under `Experiment_A/analysis/`.
- **Reproducibility caveat:** On fresh Colab runtimes, dependency availability and the behavior of pinned historical software builds may differ. Compare regenerated output metrics with the saved CSVs before claiming a complete successful rerun.